In [ ]:
# imports
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
import sys
import os
import umap

sys.path.append(os.path.abspath('..'))

from read_db import loadAlbumsDf

In [2]:
# load dataframe
df = loadAlbumsDf("/media/shared/projects/albumify/src/backend/app/db/albumify.db")

In [3]:
# first convert release date to release years for consistency
def parseReleaseYear(date):
    if pd.isna(date):
        return None
    parsed = pd.to_datetime(date, errors='coerce')
    if pd.notna(parsed):
        return parsed.year
    return int(str(date)[:4])

df['release_year'] = df['release_date'].apply(parseReleaseYear)
df.drop(columns=['release_date'], inplace=True)

In [4]:
# then multi-hot encode genres
mlbGenres = MultiLabelBinarizer()
genreMatrix = mlbGenres.fit_transform(df['genres'].apply(lambda x: x if x else []))
genre_df = pd.DataFrame(genreMatrix, columns=[f"genre_{g}" for g in mlbGenres.classes_])

In [5]:
# then multi-hot encode tags 
all_tags = set(t['tag'] for tags in df['album_tags'].dropna() for t in tags)

def buildTagVector(tags, all_tags):
    vec = {f"atag_{t}": 0 for t in all_tags}
    if tags:
        for t in tags:
            vec[f"atag_{t['tag']}"] = t['weight']
    return vec

album_tag_df = pd.DataFrame(df['album_tags'].apply(lambda x: buildTagVector(x, all_tags)).tolist())

In [6]:
all_tags = set(t['tag'] for tags in df['artist_tags'].dropna() for t in tags)
artist_tag_df = pd.DataFrame(df['artist_tags'].apply(lambda x: buildTagVector(x, all_tags)).tolist())

In [7]:
# drop columns not needed for ml
df_ml = df.drop(columns=['album_id', 'album_name', 'artist_names', 
                          'genres', 'album_tags', 'artist_tags'])

# combine with encoded columns
df_ml = pd.concat([df_ml, genre_df, artist_tag_df, album_tag_df], axis=1)

# scale everything
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df_ml), columns=df_ml.columns)

so far, all current features are 'ready' for training (i.e no nulls and all numeric)

**HOWEVER**, the feature space is made massive by the multihot encoding of tags, so UMAP will be needed later

Also some feature engineering is going to be needed

In [15]:
# feature engineering
# creating decade feature
df['decade'] = (df['release_year'] // 10 * 10).astype(int)

In [ ]:
# one hot encode
decade_dummies = pd.get_dummies(df['decade'], prefix='decade')
df_ml = pd.concat([df_ml, decade_dummies], axis=1)

In [19]:
df_ml

,album_popularity,avg_track_duration,avg_artist_popularity,release_year,genre_acid jazz,genre_acid rock,genre_adult standards,genre_afro-cuban jazz,genre_alternative dance,genre_alternative hip hop,...,atag_lp,atag_surf rock,decade_1950,decade_1960,decade_1970,decade_1980,decade_1990,decade_2000,decade_2010,decade_2020
0,35,258135.600000,68.0,1985,0,0,0,0,0,0,...,0,0,False,False,False,True,False,False,False,False
1,33,582350.000000,56.0,1974,0,0,0,0,0,0,...,0,0,False,False,True,False,False,False,False,False
2,71,287046.250000,67.0,1984,0,0,0,0,0,0,...,0,0,False,False,False,True,False,False,False,False
3,64,187491.333333,83.0,2018,0,0,0,0,0,0,...,0,0,False,False,False,False,False,False,True,False
4,0,235760.900000,60.0,1970,0,0,0,0,0,0,...,0,0,False,False,True,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
691,41,272219.483871,72.0,1970,0,1,0,0,0,0,...,0,0,False,False,True,False,False,False,False,False
692,78,311702.727273,74.0,1970,0,0,0,0,0,0,...,0,0,False,False,True,False,False,False,False,False
693,75,296225.562500,89.0,2015,0,0,0,0,0,0,...,0,0,False,False,False,False,False,False,True,False
694,43,240086.300000,67.0,1970,0,1,0,0,0,0,...,0,0,False,False,True,False,False,False,False,False
